# Posterior-Guided BDDM / Splitting-Induced Perturbation Diagnostics

This Colab notebook runs closed-form CUDA/PyTorch diagnostics for posterior-guided BDDM splitting. It does not train neural networks.

## 0_check_cuda

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
torch.set_default_dtype(torch.float64)

## 1_install_and_import

In [ ]:
from pathlib import Path
import os, sys

REPO_URL = "https://github.com/Seif-Hussein/blind-diffusion-toy.git"
BRANCH = "agent/colab-cuda-oracle"
ROOT = Path("/content/blind_splitting_bddm")

if not ROOT.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {ROOT}
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working directory:", Path.cwd())
!python -m pip install -q -r blind_splitting_bddm/requirements.txt

In [ ]:
import torch, numpy as np, pandas as pd
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2_run_smoke_test

This runs the requested reduced closed-form validation path: scale inference, posterior correction oracle, PDHG dual law, and a reduced closed-loop hierarchy.

In [ ]:
!python -m blind_splitting_bddm.src.runners.run_smoke \
  --config blind_splitting_bddm/configs/smoke.yaml \
  --device auto \
  --save_dir blind_splitting_bddm/results/smoke

## 3_run_one_step_oracle_tests

Run this if you want the posterior-correction oracle sweep by itself.

In [ ]:
!python -m blind_splitting_bddm.src.runners.run_posterior_correction_tests \
  --config blind_splitting_bddm/configs/posterior_correction_oracle.yaml \
  --device auto \
  --save_dir blind_splitting_bddm/results/posterior_correction

## 4_run_dual_law_tests

This verifies the quadratic PDHG dual-memory identity and reports bias/anisotropy of the induced perturbation.

In [ ]:
!python -m blind_splitting_bddm.src.runners.run_dual_law_tests \
  --config blind_splitting_bddm/configs/dual_law_pdhg_quadratic.yaml \
  --device auto \
  --save_dir blind_splitting_bddm/results/dual_law

## 5_run_closed_loop_hierarchy

This is the reduced hierarchy: posterior oracle, exact `c_star`, blind split, naive force, scheduled split, posterior-scale split, and raw HQS.

In [ ]:
!python -m blind_splitting_bddm.src.runners.run_closed_loop \
  --config blind_splitting_bddm/configs/closed_loop_reduced.yaml \
  --device auto \
  --save_dir blind_splitting_bddm/results/closed_loop_reduced

## 6_generate_report

In [ ]:
!python -m blind_splitting_bddm.src.runners.make_report \
  --config blind_splitting_bddm/configs/smoke.yaml \
  --save_dir blind_splitting_bddm/results/smoke

from pathlib import Path
report = Path("blind_splitting_bddm/results/smoke/combined_report.md")
print(report.read_text() if report.exists() else "Combined report not found")

In [ ]:
from pathlib import Path
for p in sorted(Path("blind_splitting_bddm/results").rglob("*.md")):
    print(p)
!zip -qr blind_splitting_bddm_results.zip blind_splitting_bddm/results
print("Created blind_splitting_bddm_results.zip")